In [1]:
from pathlib import Path
import os
import cv2
import numpy as np

def process_and_save_annotated_image(image_path, output_dir="images/annotated"):
    # Load image
    image_path = "images/boost/" + image_path
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")

    h, w, _ = img.shape
    center_x, center_y = w // 2, h // 2

    # 1. Define inner and outer radii for the ring mask
    outer_radius = int(min(w, h) * 0.55)
    inner_radius = int(min(w, h) * 0.42)

    ring_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(ring_mask, (center_x, center_y), outer_radius, 255, -1)
    cv2.circle(ring_mask, (center_x, center_y), inner_radius, 0, -1)

    # 2. Convert to HSV and segment bright green pixels
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower_green = np.array([35, 120, 120])
    upper_green = np.array([85, 255, 255])
    green_mask = cv2.inRange(hsv, lower_green, upper_green)

    # 3. Calculate boost proportion
    green_ring = cv2.bitwise_and(green_mask, green_mask, mask=ring_mask)
    total_ring_pixels = cv2.countNonZero(ring_mask)
    green_ring_pixels = cv2.countNonZero(green_ring)
    proportion = (green_ring_pixels / total_ring_pixels) if total_ring_pixels > 0 else 0.0

    # 4. Create annotated overlay image
    annotated = img.copy()

    # Draw outer and inner ring boundaries in blue
    cv2.circle(annotated, (center_x, center_y), outer_radius, (255, 0, 0), 1)
    cv2.circle(annotated, (center_x, center_y), inner_radius, (255, 0, 0), 1)

    # Tint detected green pixels in red overlay to show detection accuracy
    annotated[green_ring > 0] = [0, 0, 255] 

    # 5. Save annotated image to output directory
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.basename(image_path)
    output_path = os.path.join(output_dir, filename)
    cv2.imwrite(output_path, annotated)

    return proportion, output_path

# Example execution
image_files = list(os.listdir("./images/boost"))
print(image_files)
image_files = list(filter(lambda x: "_boost" in x, image_files))
for filename in image_files:
    try:
        prop, saved_path = process_and_save_annotated_image(filename)
        print(f"Processed: {filename} -> Fill: {prop:.2f} | Saved: {saved_path}")
    except Exception as e:
        print(f"Error processing {filename}: {e}")

['boost_it107_full_tank260.png', 'boost_it100_boost974.png', 'boost_it100_full_tank10.png', 'boost_it100_boost650.png', 'boost_it100_boost6673.png', 'boost_it100_full_tank.png', 'boost_it100_boost.png', 'boost_it100_boost88.png', 'boost_it107_boost885.png', 'boost_it107_boost80.png', 'boost_it100_boost65.png', 'boost_it100_boost6340.png']
Processed: boost_it100_boost974.png -> Fill: 0.00 | Saved: images/annotated/boost_it100_boost974.png
Processed: boost_it100_boost650.png -> Fill: 0.00 | Saved: images/annotated/boost_it100_boost650.png
Processed: boost_it100_boost6673.png -> Fill: 0.00 | Saved: images/annotated/boost_it100_boost6673.png
Processed: boost_it100_boost.png -> Fill: 0.65 | Saved: images/annotated/boost_it100_boost.png
Processed: boost_it100_boost88.png -> Fill: 0.40 | Saved: images/annotated/boost_it100_boost88.png
Processed: boost_it107_boost885.png -> Fill: 0.45 | Saved: images/annotated/boost_it107_boost885.png
Processed: boost_it107_boost80.png -> Fill: 0.59 | Saved: i

In [3]:
from pathlib import Path
import os
import cv2
import numpy as np

def process_and_save_annotated_image(image_path, output_dir="images/annotated"):
    # Load image
    full_path = os.path.join("images/boost", image_path)
    img = cv2.imread(full_path)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {full_path}")

    h, w, _ = img.shape
    center_x, center_y = w // 2, h // 2

    # 1. Define inner and outer radii for the ring mask (UNCHANGED)
    outer_radius = int(min(w, h) * 0.55)
    inner_radius = int(min(w, h) * 0.42)

    ring_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(ring_mask, (center_x, center_y), outer_radius, 255, -1)
    cv2.circle(ring_mask, (center_x, center_y), inner_radius, 0, -1)

    # 2. Convert to HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # --- COLOR MASK TUNING ---
    # Detect Green Fill: standard bright green hue range
    lower_green = np.array([35, 100, 100])
    upper_green = np.array([85, 255, 255])
    mask_green = cv2.inRange(hsv, lower_green, upper_green)

    # Detect Red Fill: Red spans hue 0-10 and 170-180 in OpenCV HSV
    lower_red1 = np.array([0, 120, 120])
    upper_red1 = np.array([10, 255, 255])
    lower_red2 = np.array([170, 120, 120])
    upper_red2 = np.array([180, 255, 255])
    mask_red = cv2.inRange(hsv, lower_red1, upper_red1) | cv2.inRange(hsv, lower_red2, upper_red2)

    # Detect Bright/White Fill: low saturation, high value (brightness)
    lower_white = np.array([0, 0, 200])
    upper_white = np.array([180, 60, 255])
    mask_white = cv2.inRange(hsv, lower_white, upper_white)

    # Combine all active filled states (Green, Red, Bright/White)
    active_fill_mask = cv2.bitwise_or(mask_green, mask_red)
    active_fill_mask = cv2.bitwise_or(active_fill_mask, mask_white)

    # 3. Calculate boost proportion on ring
    boost_ring = cv2.bitwise_and(active_fill_mask, active_fill_mask, mask=ring_mask)
    total_ring_pixels = cv2.countNonZero(ring_mask)
    boost_ring_pixels = cv2.countNonZero(boost_ring)
    proportion = (boost_ring_pixels / total_ring_pixels) if total_ring_pixels > 0 else 0.0

    # 4. Create annotated overlay image
    annotated = img.copy()

    # Draw outer and inner ring boundaries in blue
    cv2.circle(annotated, (center_x, center_y), outer_radius, (255, 0, 0), 1)
    cv2.circle(annotated, (center_x, center_y), inner_radius, (255, 0, 0), 1)

    # Highlight detected active boost pixels in bright cyan for visibility
    annotated[boost_ring > 0] = [255, 255, 0] 

    # 5. Save annotated image
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.basename(image_path)
    output_path = os.path.join(output_dir, filename)
    cv2.imwrite(output_path, annotated)

    return proportion, output_path

# Example execution
image_dir = "./images/boost"
if os.path.exists(image_dir):
    image_files = list(os.listdir(image_dir))
    image_files = list(filter(lambda x: "_boost" in x, image_files))
    for filename in image_files:
        try:
            prop, saved_path = process_and_save_annotated_image(filename)
            print(f"Processed: {filename} -> Fill: {prop:.2f} | Saved: {saved_path}")
        except Exception as e:
            print(f"Error processing {filename}: {e}")

Processed: boost_it100_boost974.png -> Fill: 0.08 | Saved: images/annotated/boost_it100_boost974.png
Processed: boost_it100_boost4775.png -> Fill: 0.52 | Saved: images/annotated/boost_it100_boost4775.png
Processed: boost_it100_boost650.png -> Fill: 0.00 | Saved: images/annotated/boost_it100_boost650.png
Processed: boost_it100_boost6673.png -> Fill: 0.08 | Saved: images/annotated/boost_it100_boost6673.png
Processed: boost_it100_boost.png -> Fill: 0.75 | Saved: images/annotated/boost_it100_boost.png
Processed: boost_it100_boost88.png -> Fill: 0.52 | Saved: images/annotated/boost_it100_boost88.png
Processed: boost_it107_boost885.png -> Fill: 0.56 | Saved: images/annotated/boost_it107_boost885.png
Processed: boost_it107_boost80.png -> Fill: 0.77 | Saved: images/annotated/boost_it107_boost80.png
Processed: boost_it100_boost65.png -> Fill: 0.54 | Saved: images/annotated/boost_it100_boost65.png
Processed: boost_it100_boost6340.png -> Fill: 0.87 | Saved: images/annotated/boost_it100_boost6340.

In [25]:
from modules.frame import get_window_region

region = get_window_region("BlueStacks")
jbx = region["left"] + int(region["width"] * 0.7)
jby = region["top"]  + int(region["height"] * 0.85)
print("clicking jump button at", jbx, jby)
import pyautogui, time
def click_fn():
    pyautogui.mouseDown(jbx, jby)
    time.sleep(0.03)          # hold briefly so BlueStacks registers a touch
    pyautogui.mouseUp(jbx, jby)

click_fn()

clicking jump button at 385 932
